# Import Libraries

In [6]:
import pandas as pd
import numpy as np
import collections

# Import DataFrame

In [7]:
sampled_df_filepath = "sampled_dataframe.pkl"

sampled_df = pd.read_pickle(sampled_df_filepath).reset_index(drop=True)

### Getting Unique Vocabulary

In [8]:
vocab = set()

for sublist in sampled_df['lemmas']:
    vocab.update(sublist)

vocab_size = len(vocab)

print(f"Vocab size is {vocab_size} lemmas.")

Vocab size is 159862 lemmas.


### Generating BoW Representations

The following cell determines the minimum vocabulary size needed to cover 80% of all token occurrences in the corpus by calculating cumulative relative word frequencies.

In [9]:
token_frequencies = collections.Counter()

for sentence in sampled_df["lemmas"]:
    token_frequencies.update(sentence)

sorted_counts = np.sort(np.fromiter(token_frequencies.values(), dtype=np.uint32), descending=True)
cumulative_sorted_counts = np.cumsum(sorted_counts)
normalized_cumulative_sorted_counts = cumulative_sorted_counts / cumulative_sorted_counts[-1]

FREQUENCY_THRESHOLD = 0.8
threshold = np.searchsorted(normalized_cumulative_sorted_counts, FREQUENCY_THRESHOLD) + 1

print(f"We will use the most frequent {threshold} lemmas as our vocab and insert an additional unknown character (<UNK>) for everything else.")

We will use the most frequent 1126 lemmas as our vocab and insert an additional unknown character (<UNK>) for everything else.


In [10]:
top_lemmas_with_count = token_frequencies.most_common(threshold)
vocab = [lemma for lemma, _ in top_lemmas_with_count]

vocab_size = len(vocab) + 1  # 1 for <UNK> (vocab_size - 1)
print(f"Vocab size is {vocab_size} lemmas.")

# 1. Build the mapping dictionary
word_to_id = {word: idx for idx, word in enumerate(vocab)}
word_to_id['<UNK>'] = vocab_size - 1

# 2. Build BoW array
bow_representations = np.zeros((sampled_df.shape[0], vocab_size), dtype=np.uint16)
unk_id = word_to_id['<UNK>']

for i, sentence in enumerate(sampled_df['lemmas']):
    for lemma in sentence:
        bow_representations[i, word_to_id.get(lemma, unk_id)] += 1

Vocab size is 1127 lemmas.


In [36]:
sampled_df['lemmas']

0         [utóbbi, szintén, az, elmúlt, időszak, gyakorl...
1         [mindenképpen, el, tud, mond, ,, hogy, az, lmp...
2         [ez, szeret, megcéloz, ,, hogy, 100, százaléko...
3         [a, szokatlan, magas, részvételi, arány, pedig...
4         [a, maradandó, értékű, kárpótlási, irat, levél...
                                ...                        
509955    [munkaerőpiaci, szempont, 600, ezer, igen, fon...
509956    [és, az, is, megmond, ön, ,, hogy, mi, ez, az,...
509957    [mi, pedig, az, európai, uniós, kérdés, illet,...
509958    [az, a, folyamat, ,, ami, elindul, a, roma, in...
509959    [a, harmadik, jogszabály, ,, az, egyes, közszo...
Name: lemmas, Length: 509960, dtype: object

In [ ]:
class BoWEncoder():
    def __init__(self, freq_threshold=0.8):
        self.freq_threshold = freq_threshold
        self.word_to_id = {}
        self.vocab = []
        self.vocab_size = 0
        self.unk_id = None

    def _calculate_threshold(self, token_frequencies):
        counts = np.array([count for _, count in token_frequencies.most_common()], dtype=np.uint64)
        cumulative_counts = np.cumsum(counts)
        normalized_cumulative_counts = cumulative_counts / cumulative_counts[-1]

        return int(np.searchsorted(normalized_cumulative_counts, self.freq_threshold) + 1)

    def fit(self, sentences):
        token_frequencies = collections.Counter()
        for sentence in sentences:
            token_frequencies.update(sentence)

        threshold = self._calculate_threshold(token_frequencies)
        top_lemmas_with_count = token_frequencies.most_common(threshold)

        self.vocab = [lemma for lemma, _ in top_lemmas_with_count]
        self.unk_id = len(self.vocab)
        self.vocab_size = len(self.vocab) + 1  # 1 for <UNK> (vocab_size - 1)

        self.word_to_id = {word: idx for idx, word in enumerate(self.vocab)}
        self.word_to_id['<UNK>'] = self.unk_id

        return self

    def transform(self, sentences):
        if not self.word_to_id:
            raise RuntimeError("BoWEncoder must be fitted before calling transform().")

        num_samples = len(sentences)
        bow_representations = np.zeros((num_samples, self.vocab_size), dtype=np.uint16)

        word_mapping = self.word_to_id
        unk_idx = self.unk_id

        for i, sentence in enumerate(sentences):
            for lemma in sentence:
                bow_representations[i, word_mapping.get(lemma, unk_idx)] += 1

        return bow_representations

    def fit_transform(self, sentences):
        return self.fit(sentences).transform(sentences)

# Usage:
# encoder = BoWEncoder(freq_threshold=0.8)
# X_train_bow = encoder.fit_transform(train_df['lemmas'])
# X_test_bow = encoder.transform(test_df['lemmas'])

# 1D Convolutions

Creating dictionary from vocab

In [11]:
word_to_id = {word: idx for idx, word in enumerate(vocab, start=1)}
word_to_id['<PAD>'] = 0
word_to_id['<UNK>'] = vocab_size - 1

Generating int representations

In [12]:
max_length = 50
num_samples = sampled_df.shape[0]
int_representations = np.zeros((num_samples, max_length), dtype=np.uint16)
unk_id = word_to_id['<UNK>']

for (i, sentence) in enumerate(sampled_df['lemmas']):
    sentence = sentence[:max_length]

    for (j, lemma) in enumerate(sentence):
        int_representations[i, j] = word_to_id.get(lemma, unk_id)

In [87]:
one_hot_representations = np.eye(vocab_size)[int_representations]

: 

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

le = LabelEncoder()

target_column = 'Party_orientation'
y = le.fit_transform(sampled_df[target_column])
X = int_representations

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

Defining Model Architecture

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvNet(nn.Module):
    def __init__(self, vocab_size, num_classes=9, output_channels=64, kernel_size=3):
        super(ConvNet, self).__init__()
        self.conv1 = nn.Conv1d(vocab_size, output_channels, kernel_size)
        self.pool = nn.AdaptiveMaxPool1d(output_size=1)
        self.fc1 = nn.Linear(output_channels, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1) # (batch, max_length, vocab_size) -> (batch, vocab_size, max_length)

        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)

        x = x.squeeze(-1) # (batch, 64, 1) -> (batch, 64)
        x = self.fc1(x)

        return x

In [25]:
y.shape

(509960,)

In [26]:
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_tensor_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_tensor_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 128
train_loader = DataLoader(train_tensor_dataset, batch_size, shuffle=True)
test_loader = DataLoader(test_tensor_dataset, batch_size, shuffle=True)

In [28]:
import torch.optim as optim

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = ConvNet(vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Test for one batch

In [29]:
batch_x, batch_y = next(iter(train_loader))

batch_x = F.one_hot(batch_x, num_classes=vocab_size).float()

batch_x = batch_x.to(device)
batch_y = batch_y.to(device)

model.train()
for step in range(40):
    optimizer.zero_grad()
    outputs = model(batch_x)
    loss = criterion(outputs, batch_y)
    loss.backward()
    optimizer.step()
    
    if (step + 1) % 10 == 0:
        print(f"Step {step+1}, Loss: {loss.item():.4f}")

Step 10, Loss: 2.0893
Step 20, Loss: 2.0008
Step 30, Loss: 1.8877
Step 40, Loss: 1.7548


In [33]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

num_epochs = 5

for epoch in range(num_epochs):
    # -----------------------------
    # 1. TRAINING PHASE
    # -----------------------------
    model.train()
    
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    # Wrap the train_loader with tqdm
    train_pbar = tqdm(
        train_loader, 
        desc=f"Epoch {epoch + 1}/{num_epochs} [Train]", 
        leave=False
    )

    for batch_x, batch_y in train_pbar:
        # On-the-fly one-hot encoding & device placement
        batch_x = F.one_hot(batch_x, num_classes=vocab_size).float().to(device)
        batch_y = batch_y.to(device)

        # Optimization step
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        # Running metrics tracking
        batch_size = batch_y.size(0)
        running_train_loss += loss.item() * batch_size
        predictions = outputs.argmax(dim=1)
        correct_train += (predictions == batch_y).sum().item()
        total_train += batch_size

        # Update live progress bar display
        current_avg_loss = running_train_loss / total_train
        current_acc = (correct_train / total_train) * 100
        train_pbar.set_postfix({
            "batch_loss": f"{loss.item():.4f}",
            "avg_loss": f"{current_avg_loss:.4f}",
            "acc": f"{current_acc:.2f}%"
        })

    epoch_train_loss = running_train_loss / total_train
    epoch_train_acc = correct_train / total_train

    # -----------------------------
    # 2. VALIDATION PHASE
    # -----------------------------
    model.eval()
    
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    val_pbar = tqdm(
        test_loader, 
        desc=f"Epoch {epoch + 1}/{num_epochs} [Val]", 
        leave=False
    )

    with torch.no_grad():
        for batch_x, batch_y in val_pbar:
            batch_x = F.one_hot(batch_x, num_classes=vocab_size).float().to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)

            batch_size = batch_y.size(0)
            running_val_loss += loss.item() * batch_size
            predictions = outputs.argmax(dim=1)
            correct_val += (predictions == batch_y).sum().item()
            total_val += batch_size

            val_pbar.set_postfix({
                "val_loss": f"{loss.item():.4f}",
                "val_acc": f"{(correct_val / total_val) * 100:.2f}%"
            })

    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = correct_val / total_val

    # -----------------------------
    # 3. EPOCH SUMMARY
    # -----------------------------
    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc * 100:.2f}% | "
        f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc * 100:.2f}%"
    )

/Users/mikhail/Documents/Masters/NLP 1/project/nlp-project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch [1/5] | Train Loss: 1.4563 - Train Acc: 44.88% | Val Loss: 1.4997 - Val Acc: 42.75%


Epoch [2/5] | Train Loss: 1.4141 - Train Acc: 46.78% | Val Loss: 1.4994 - Val Acc: 42.98%


Epoch [3/5] | Train Loss: 1.3812 - Train Acc: 48.22% | Val Loss: 1.5073 - Val Acc: 42.65%


Epoch [4/5] | Train Loss: 1.3540 - Train Acc: 49.43% | Val Loss: 1.5203 - Val Acc: 42.24%


Epoch [5/5] | Train Loss: 1.3320 - Train Acc: 50.35% | Val Loss: 1.5300 - Val Acc: 41.91%


Lets make the model deeper

In [34]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvNet(nn.Module):
    def __init__(self, vocab_size, num_classes=9, output_channels=64, kernel_size=3):
        super(ConvNet, self).__init__()
        self.conv1 = nn.Conv1d(vocab_size, output_channels, kernel_size, padding='same')
        self.conv2 = nn.Conv1d(output_channels, output_channels, kernel_size, padding='same')
        self.conv3 = nn.Conv1d(output_channels, output_channels, kernel_size, padding='same')
        self.pool = nn.AdaptiveMaxPool1d(output_size=1)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(output_channels, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1) # (batch, max_length, vocab_size) -> (batch, vocab_size, max_length)

        x = self.conv1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = F.relu(x)

        x = self.conv3(x)
        x = F.relu(x)

        x = self.pool(x)


        x = x.squeeze(-1) # (batch, 64, 1) -> (batch, 64)

        x = self.dropout(x)

        x = self.fc1(x)

        return x

In [35]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

num_epochs = 5

for epoch in range(num_epochs):
    # -----------------------------
    # 1. TRAINING PHASE
    # -----------------------------
    model.train()
    
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    # Wrap the train_loader with tqdm
    train_pbar = tqdm(
        train_loader, 
        desc=f"Epoch {epoch + 1}/{num_epochs} [Train]", 
        leave=False
    )

    for batch_x, batch_y in train_pbar:
        # On-the-fly one-hot encoding & device placement
        batch_x = F.one_hot(batch_x, num_classes=vocab_size).float().to(device)
        batch_y = batch_y.to(device)

        # Optimization step
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        # Running metrics tracking
        batch_size = batch_y.size(0)
        running_train_loss += loss.item() * batch_size
        predictions = outputs.argmax(dim=1)
        correct_train += (predictions == batch_y).sum().item()
        total_train += batch_size

        # Update live progress bar display
        current_avg_loss = running_train_loss / total_train
        current_acc = (correct_train / total_train) * 100
        train_pbar.set_postfix({
            "batch_loss": f"{loss.item():.4f}",
            "avg_loss": f"{current_avg_loss:.4f}",
            "acc": f"{current_acc:.2f}%"
        })

    epoch_train_loss = running_train_loss / total_train
    epoch_train_acc = correct_train / total_train

    # -----------------------------
    # 2. VALIDATION PHASE
    # -----------------------------
    model.eval()
    
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    val_pbar = tqdm(
        test_loader, 
        desc=f"Epoch {epoch + 1}/{num_epochs} [Val]", 
        leave=False
    )

    with torch.no_grad():
        for batch_x, batch_y in val_pbar:
            batch_x = F.one_hot(batch_x, num_classes=vocab_size).float().to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)

            batch_size = batch_y.size(0)
            running_val_loss += loss.item() * batch_size
            predictions = outputs.argmax(dim=1)
            correct_val += (predictions == batch_y).sum().item()
            total_val += batch_size

            val_pbar.set_postfix({
                "val_loss": f"{loss.item():.4f}",
                "val_acc": f"{(correct_val / total_val) * 100:.2f}%"
            })

    epoch_val_loss = running_val_loss / total_val
    epoch_val_acc = correct_val / total_val

    # -----------------------------
    # 3. EPOCH SUMMARY
    # -----------------------------
    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc * 100:.2f}% | "
        f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc * 100:.2f}%"
    )

Epoch [1/5] | Train Loss: 1.3134 - Train Acc: 51.21% | Val Loss: 1.5430 - Val Acc: 41.54%


Epoch [2/5] | Train Loss: 1.2978 - Train Acc: 51.82% | Val Loss: 1.5575 - Val Acc: 41.45%


Epoch [3/5] | Train Loss: 1.2843 - Train Acc: 52.41% | Val Loss: 1.5729 - Val Acc: 40.81%


Epoch [4/5] | Train Loss: 1.2732 - Train Acc: 52.92% | Val Loss: 1.5833 - Val Acc: 41.08%


Epoch [5/5] | Train Loss: 1.2633 - Train Acc: 53.34% | Val Loss: 1.5948 - Val Acc: 40.71%
